In [ ]:
# test
import time
import re
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.select import Select

user_ID = "ckm00285"
user_PW = "cth620700!"

# ✅ performance 로그 켜기

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

driver.get("https://domesin.com/scm/login.html")

# 로그인
ID = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "body > div.login-box > form > input[type=text]:nth-child(4)")))
ID.clear()
ID.send_keys(user_ID)

PW = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "body > div > form > input[type=password]:nth-child(5)")))
PW.clear()
PW.send_keys(user_PW)

login_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > div > form > button.login-btn")))
login_btn.click()

# 주문관리 페이지 이동
order_page = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > table > tbody > tr > td:nth-child(1) > ul:nth-child(5) > li:nth-child(1) > a")))
driver.execute_script("arguments[0].click();", order_page)

# 신규주문 선택
new_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(2) > td.cttd > input[type=radio]:nth-child(2)")))
driver.execute_script('arguments[0].click()',new_order)

# 100개씩 출력
select_tag = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(5) > td.cttd > select:nth-child(2)")))
Select(select_tag).select_by_visible_text("100개씩 출력")

# 선택주문 체크박스
select_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'#main > table.mytable2 > tbody > tr:nth-child(1) > td:nth-child(1) > input[type=checkbox]')))
driver.execute_script('arguments[0].click()',select_order)

# 선택주문 확인
select_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'#main > div > input:nth-child(1)')))
driver.execute_script('arguments[0].click()',select_order)

# 알림 확인
alert = WebDriverWait(driver, 10).until(EC.alert_is_present())
print("✅ alert 문구:", alert.text)
alert.accept()

time.sleep(1.0)

# 전체주문 선택
total_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(2) > td.cttd > input[type=radio]:nth-child(1)")))
total_order.click()

# 100개씩 출력
select_tag = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(5) > td.cttd > select:nth-child(2)")))
Select(select_tag).select_by_visible_text("100개씩 출력")

# 주문검색 클릭
order_search = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(5) > td.cttd > input.mybt02")))
driver.execute_script("arguments[0].click();", order_search)
time.sleep(1)


# ✅ 다운로드 폴더(필요하면 너 환경에 맞게 수정)
DOWNLOAD_DIR = Path.home() / "Downloads"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)  # 폴더 없으면 생성
print("DOWNLOAD_DIR =", DOWNLOAD_DIR)

# ----------------------------
# 1) 주문 그룹핑 (너가 이미 성공한 버전)
# ----------------------------
def is_header_tr(tr) -> bool:
    if tr.find("th"):
        return True
    txt = tr.get_text(" ", strip=True)
    return ("주문번호" in txt and "수취인명" in txt) or ("상품명" in txt and "공급가" in txt)

def is_customerinfo_tr(tr) -> bool:
    txt = tr.get_text(" ", strip=True)
    return ("연락처1" in txt) and ("주소" in txt)

def is_main_tr(tr) -> bool:
    txt = tr.get_text(" ", strip=True)
    has_date = re.search(r"\d{4}-\d{2}-\d{2}", txt) is not None
    has_order_no = re.search(r"\b20\d{10,}\b", txt) is not None
    return has_date or has_order_no

def parse_orders_grouping(html: str, table_css="table.mytable2"):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.select_one(table_css)
    if not table:
        raise ValueError("table.mytable2 를 못 찾음")

    trs = table.select("tbody > tr")

    orders = []
    current_items = []

    for tr in trs:
        if is_header_tr(tr):
            continue

        if is_customerinfo_tr(tr):
            cust_txt = tr.get_text(" ", strip=True)
            if current_items:
                orders.append({
                    "items": current_items,
                    "customer_info_text": cust_txt
                })
                current_items = []
            continue

        if is_main_tr(tr):
            current_items.append(tr)
            continue

    return orders

# ----------------------------
# 2) 필드 추출 함수들
# ----------------------------
def extract_order_no(tr):
    """
    주문번호는 a태그가 아니라 div onclick="view_order('...')" 안에 있음
    """
    d = tr.select_one("div[onclick*='view_order']")
    if not d:
        return ""
    txt = d.get_text(strip=True)
    # 화면에 보이는 주문번호가 긴 숫자면 그걸 우선
    if re.fullmatch(r"\d{10,}", txt):
        return txt
    # fallback: onclick에서 내부 id라도 추출
    oc = d.get("onclick", "")
    m = re.search(r"view_order\('(\d+)'\)", oc)
    return m.group(1) if m else txt

def extract_receiver_from_first_tr(first_tr):
    """
    수취인명은 bold style td로 잡힘 (길이 제한 없음)
    """
    td = first_tr.select_one("td[style*='font-weight:bold']")
    if td:
        return td.get_text(" ", strip=True)
    return ""

def extract_supply_price(first_tr):
    """
    1) first_tr에서 '상품정보(가장 긴 td)' 칸을 찾고
    2) 그 다음 td들 중 '숫자만 있는 첫 칸'을 공급가로 반환
    """
    tds = [td.get_text(" ", strip=True) for td in first_tr.find_all("td", recursive=False)]
    if not tds:
        return ""

    # 상품정보 덩어리 = 가장 긴 텍스트 td
    idx_prod = max(range(len(tds)), key=lambda i: len(tds[i]))

    # 상품정보 다음부터 숫자-only 찾기 (공급가)
    for s in tds[idx_prod + 1:]:
        x = s.replace(",", "").replace(" ", "")
        if x.isdigit():
            return x  # 문자열로 반환(엑셀 저장용)
    return ""

def extract_product_blob(first_tr):
    """
    상품정보는 td 중 가장 긴 텍스트인 경우가 대부분이라서 그걸 사용
    """
    tds = [td.get_text(" ", strip=True) for td in first_tr.find_all("td", recursive=False)]
    return max(tds, key=len) if tds else ""

# ✅ 새롭게 추가된 코드: TS코드/업체코드/상품명/택배송장명/옵션명 통합 추출
# ✅ 새롭게 추가된 코드: 상품정보_raw(한 줄 텍스트)에서 구간으로 정확히 추출
def extract_product_pack(product_text: str):
    """
    product_blob(상품정보 td 텍스트)에서:
    - ts_code
    - vendor_code (KM_A069/KM_A070 같은 슬래시 포함도 OK)
    - product_name (업체상품코드 뒤 ~ (택배송장명...) 앞)
    - invoice_name (괄호 안 택배송장명)
    - option_name (')' 뒤 ~ '택배업체선택' 앞)
    """
    if not product_text:
        return "", "", "", "", ""

    # 공백 정리 (NBSP 포함)
    flat = product_text.replace("\xa0", " ")
    flat = re.sub(r"\s+", " ", flat).strip()

    # 1) TS 상품코드
    m_ts = re.search(r"\bTS\d+\b", flat)
    ts_code = m_ts.group(0) if m_ts else ""

    # 2) 업체상품코드: 슬래시 포함 허용 (KM_A069/KM_A070)
    m_vendor = re.search(
        r"업체상품코드\s*[:：]?\s*([A-Z]{2}[_-]?[A-Z0-9]+(?:/[A-Z]{2}[_-]?[A-Z0-9]+)*)",
        flat
    )
    vendor_code = m_vendor.group(1) if m_vendor else ""

    # 3) 택배송장명(괄호 안)
    m_inv = re.search(r"\(\s*택배송장명\s*[:：]\s*([^)]+?)\s*\)", flat)
    invoice_name = m_inv.group(1).strip() if m_inv else ""

    product_name = ""
    option_name = ""

    # 4) 상품명 / 옵션명은 '업체상품코드' ~ '(택배송장명' ~ '택배업체선택' 구간으로 자르기
    if vendor_code and m_inv:
        start = flat.find(vendor_code)
        if start != -1:
            start += len(vendor_code)

            inv_pos = flat.find(m_inv.group(0), start)  # "(택배송장명: ...)" 전체 문자열 위치
            if inv_pos != -1:
                # 상품명 = vendor_code 뒤 ~ (택배송장명...) 앞
                product_name = flat[start:inv_pos].strip(" -/|")

                # 옵션명 = ')' 뒤 ~ '택배업체선택' 앞
                after_paren = inv_pos + len(m_inv.group(0))
                end_key = "택배업체선택"
                end_pos = flat.find(end_key, after_paren)
                if end_pos == -1:
                    option_name = flat[after_paren:].strip(" -/|")
                else:
                    option_name = flat[after_paren:end_pos].strip(" -/|")

    # 5) 상품명이 그래도 비었으면(예외 케이스 대비) fallback:
    #    업체상품코드 뒤에서 "택배업체선택" 앞까지 중, 괄호 부분 제거 후 앞쪽을 상품명으로
    if not product_name and vendor_code:
        start = flat.find(vendor_code)
        if start != -1:
            start += len(vendor_code)
            end_pos = flat.find("택배업체선택", start)
            chunk = flat[start:end_pos].strip() if end_pos != -1 else flat[start:].strip()
            # "(택배송장명: ...)" 제거
            chunk = re.sub(r"\(\s*택배송장명\s*[:：]\s*[^)]+\)", "", chunk).strip()
            # 남은 chunk에서 앞부분을 상품명으로
            if chunk:
                # 옵션이 같이 붙어있으면 길이가 긴 편이라도 일단 chunk 전체를 상품명으로 넣고
                # 옵션은 따로 못 뽑으면 빈칸으로 둠
                product_name = chunk

    return ts_code, vendor_code, product_name, invoice_name, option_name


# ✅ 새롭게 추가된 코드 (기존 extract_phones 대신 이걸로 교체)
def extract_customer_fields(customer_info_text: str):
    """
    customer_info_text 한 줄에서 아래 항목을 각각 추출해서 반환
    반환: phone1, phone2, zipcode, address, request_msg

    실제 케이스(너 스샷):
    '연락처1: ... 연락처2: ... 주소:(우:62030) 광주... 배송요청사항: ...'
    또는
    '주소:(62030) ...' / '주소: 우:62030 ...' 등도 대응
    """
    phone1 = ""
    phone2 = ""
    zipcode = ""
    address = ""
    request_msg = ""

    if not customer_info_text:
        return phone1, phone2, zipcode, address, request_msg

    txt = customer_info_text

    # 1) 연락처1/2
    m1 = re.search(r"연락처1\s*:\s*([0-9\-]{9,13})", txt)
    m2 = re.search(r"연락처2\s*:\s*([0-9\-]{9,13})", txt)
    if m1:
        phone1 = m1.group(1)
    if m2:
        phone2 = m2.group(1)

    # 2) 주소 원문만 먼저 뽑기: "주소:" 다음부터 "배송요청사항:" 전까지
    madd = re.search(r"주소\s*[:：]?\s*(.*?)(?=배송요청사항\s*[:：]|$)", txt)
    addr_raw = madd.group(1).strip() if madd else ""

    # 3) 우편번호 추출 (우:12345) / (12345) / 우:12345 모두 대응
    mzip = re.search(r"\(\s*우\s*[:：]\s*(\d{5})\s*\)", addr_raw)
    if mzip:
        zipcode = mzip.group(1)
    else:
        mzip2 = re.search(r"\(\s*(\d{5})\s*\)", addr_raw)
        if mzip2:
            zipcode = mzip2.group(1)
        else:
            mzip3 = re.search(r"\b우\s*[:：]\s*(\d{5})\b", addr_raw)
            if mzip3:
                zipcode = mzip3.group(1)

    # 4) 주소에서 우편번호 표기 제거해서 주소만 남기기
    address = addr_raw
    address = re.sub(r"^\(\s*우\s*[:：]\s*\d{5}\s*\)\s*", "", address)  # (우:12345) 제거
    address = re.sub(r"^\(\s*\d{5}\s*\)\s*", "", address)            # (12345) 제거
    address = address.strip()

    # 5)주소 끝에 붙는 "|" 제거 (공백 포함)
    address = re.sub(r"\s*\|\s*$", "", address)

    address = address.strip()

    # 6) 배송요청사항
    mreq = re.search(r"배송요청사항\s*[:：]\s*(.*)$", txt)
    if mreq:
        request_msg = mreq.group(1).strip()

    return phone1, phone2, zipcode, address, request_msg

# 수량 추출
# ✅ 새롭게 추가된 코드: 수량 추출 (파란색 b)
def extract_quantity(item_tr):
    b = item_tr.select_one("b[style*='color:blue']")
    if b:
        t = b.get_text(strip=True)
        return t if t.isdigit() else ""
    return ""

# ✅ 새롭게 추가된 코드: 상품행 판별 (수량 파란 b가 있으면 상품행)
def is_item_tr(tr) -> bool:
    return tr.select_one("b[style*='color:blue']") is not None


# 주문일시
def extract_order_datetime(first_tr):
    """
    주문일시 td는 보통 'YYYY-MM-DD' + 줄바꿈 + 'HH:MM:SS' 형태.
    td 단위로 찾고 날짜/시간을 조합해서 'YYYY-MM-DD HH:MM:SS' 반환.
    """
    date_pat = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
    time_pat = re.compile(r"\b\d{2}:\d{2}:\d{2}\b")

    for td in first_tr.find_all("td", recursive=False):
        txt = td.get_text(" ", strip=True)

        m_date = date_pat.search(txt)
        if not m_date:
            continue

        # 같은 td 안에 시간이 같이 있으면 조합
        m_time = time_pat.search(txt)
        if m_time:
            return f"{m_date.group(0)} {m_time.group(0)}"

        # 시간이 td 밖으로 분리된 예외 대비: first_tr 전체에서 시간 찾기
        m_time2 = time_pat.search(first_tr.get_text(" ", strip=True))
        if m_time2:
            return f"{m_date.group(0)} {m_time2.group(0)}"

        # 날짜만이라도 반환
        return m_date.group(0)

    return ""

# 발송완료 제외
def extract_order_status(base_tr):
    """
    base_tr(첫 상품행)에서 주문상태 텍스트 추출
    - 도매의신은 주문상태 td 안에 <b style='color:blue'>오늘</b> 같은게 같이 들어갈 수 있어서
      전체 텍스트를 합쳐서 반환
    """
    # 주문상태 td는 보통 date/time td 다음에 등장하고,
    # 내부에 '발송완료', '배송준비중' 같은 문구가 있음.
    tds = base_tr.find_all("td", recursive=False)
    for td in tds:
        txt = td.get_text(" ", strip=True)
        if "발송완료" in txt or "배송준비중" in txt or "배송마감" in txt or "취소" in txt or "교환" in txt or "반품" in txt:
            return txt
    return ""


def is_file_locked(path: Path) -> bool:
    """
    Windows에서 엑셀로 파일이 열려있으면 PermissionError가 나는 경우가 많음.
    """
    if not path.exists():
        return False
    try:
        # append 모드로 열어보기(잠김이면 PermissionError)
        with open(path, "a", encoding="utf-8"):
            pass
        return False
    except PermissionError:
        return True
    except OSError:
        # 환경에 따라 잠김이 OSError로도 날 수 있음 -> 잠김으로 취급
        return True

def make_save_path(base_dir: Path, base_name: str, ext: str = ".xlsx") -> Path:
    """
    기본: base_name.xlsx
    - 파일이 열려있으면 base_name_YYYYMMDD_HHMMSS.xlsx 로 저장
    - 파일이 없거나 닫혀있으면 base_name.xlsx 로 저장(덮어쓰기)
    """
    base_dir.mkdir(parents=True, exist_ok=True)
    base_path = base_dir / f"{base_name}{ext}"

    if is_file_locked(base_path):
        ts = time.strftime("%Y%m%d_%H%M%S")
        return base_dir / f"{base_name}_{ts}{ext}"

    return base_path

def is_not_shipped(base_tr):
    """
    발송완료 제외 필터
    """
    status = extract_order_status(base_tr)
    return "발송완료" not in status

html = driver.page_source
orders = parse_orders_grouping(html)

rows = []

for od in orders:
    if not od.get("items"):
        continue

    # ✅ 새롭게 추가된 코드: 상품행만 필터링
    item_trs = [tr for tr in od["items"] if is_item_tr(tr)]
    if not item_trs:
        continue

    # ✅ 주문 공통 필드는 첫 상품행 기준
    base_tr = item_trs[0]
    if not is_not_shipped(base_tr):
        continue
    order_no = extract_order_no(base_tr)
    receiver = extract_receiver_from_first_tr(base_tr)
    order_dt = extract_order_datetime(base_tr)

    phone1, phone2, zipcode, address, request_msg = extract_customer_fields(
        od.get("customer_info_text", "")
    )

    # ✅ 합포장 처리: 상품행마다 한 줄씩 rows에 추가
    for item_tr in item_trs:
        product_blob = extract_product_blob(item_tr)
        ts_code, vendor_code, product_name, invoice_name, option_name = extract_product_pack(product_blob)
        
        supply = extract_supply_price(item_tr)
        quantity = extract_quantity(item_tr)

        # ✅ rows.append는 반드시 여기(상품행 루프 안)
        rows.append({
            "주문코드": order_no,
            "택배사코드": "",
            "송장번호": "",
            "수취인": receiver,
            "수취인휴대폰": phone1,
            "수취인전화": phone2,
            "우편번호": zipcode,
            "주소": address,
            "상품코드": ts_code,
            "업체상품코드": vendor_code,
            "상품명": product_name,
            "택배상품명": invoice_name,
            "제조사": "케이엠트레이드",
            "선택옵션": option_name,
            "옵션관리코드": "",
            "입력옵션": "",
            "과세여부": "과세",
            "공급가": supply,
            "수량": quantity,
            "배송구분":"기본배송",
            "배송비":"3000",
            "주문일시": order_dt,
            "주문상태":"",
            "배송처리일":"",
            "주문요청사항":request_msg,
            "주문원장고유번호":"",
            "주문취소상태":"",
            "주문반품상태":"",
            "주문교환상태":"",
        })

df = pd.DataFrame(rows)

SAVE_DIR = Path.home() / "Downloads"   # 원하면 다른 폴더로 바꿔도 됨
out_path = make_save_path(SAVE_DIR, "도매의신 수집파일", ".xlsx")

df.to_excel(out_path, index=False)
print("✅ 저장 경로:", out_path)


print("✅ 주문(행) 개수:", len(df))
print("✅ 저장 경로:", out_path)
print("✅ 파일 존재:", out_path.exists())


In [6]:
# test
import time
import re
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.select import Select

user_ID = "ckm00285"
user_PW = "cth620700!"

# ✅ performance 로그 켜기

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

driver.get("https://domesin.com/scm/login.html")

# 로그인
ID = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "body > div.login-box > form > input[type=text]:nth-child(4)")))
ID.clear()
ID.send_keys(user_ID)

PW = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "body > div > form > input[type=password]:nth-child(5)")))
PW.clear()
PW.send_keys(user_PW)

login_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > div > form > button.login-btn")))
login_btn.click()

# 주문관리 페이지 이동
order_page = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "body > table > tbody > tr > td:nth-child(1) > ul:nth-child(5) > li:nth-child(1) > a")))
driver.execute_script("arguments[0].click();", order_page)

# 신규주문 선택
new_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(2) > td.cttd > input[type=radio]:nth-child(2)")))
driver.execute_script('arguments[0].click()',new_order)

# 100개씩 출력
select_tag = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(5) > td.cttd > select:nth-child(2)")))
Select(select_tag).select_by_visible_text("100개씩 출력")

# 선택주문 체크박스
select_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'#main > table.mytable2 > tbody > tr:nth-child(1) > td:nth-child(1) > input[type=checkbox]')))
driver.execute_script('arguments[0].click()',select_order)

# 선택주문 확인
select_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'#main > div > input:nth-child(1)')))
driver.execute_script('arguments[0].click()',select_order)

# 알림 확인
alert = WebDriverWait(driver, 10).until(EC.alert_is_present())
print("✅ alert 문구:", alert.text)
alert.accept()

time.sleep(1.0)

# 전체주문 선택
total_order = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(2) > td.cttd > input[type=radio]:nth-child(1)")))
total_order.click()

# 100개씩 출력
select_tag = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#main > table.mytable1 > tbody > tr:nth-child(5) > td.cttd > select:nth-child(2)")))
Select(select_tag).select_by_visible_text("100개씩 출력")

✅ alert 문구: 선택된 주문이 없습니다
